# Chapter 5 — Planning and Execution

**Book alignment:** current Chapter 5 · internal demo `Stage 04`

The shared demo package calls this **Stage 04** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can intended future work be represented, validated, scheduled, and revised without confusing a plan with runtime truth?


## Hypothesis

A plan is a falsifiable hypothesis about a route from current state to a goal. The runtime should be able to validate its structure, choose only currently ready steps, and replace only remaining intended work when evidence invalidates an assumption.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.planning import (
    Plan,
    PlanStep,
    PlanValidator,
    Replanner,
    StepScheduler,
)

## Controlled experiment: validate the intended route


In [ ]:
plan = Plan(
    goal="Fix pipe-delimited records",
    steps=(
        PlanStep("reproduce", "run_tests", expected_evidence=("pipe test fails",)),
        PlanStep(
            "inspect",
            "read_file parser.py",
            dependencies=("reproduce",),
            preconditions=("failure reproduced",),
            expected_evidence=("delimiter code seen",),
        ),
        PlanStep(
            "patch",
            "edit parser.py",
            dependencies=("inspect",),
            preconditions=("parser inspected",),
            expected_evidence=("source changed",),
        ),
    ),
)

validator = PlanValidator()
valid = validator.validate(plan)

invalid = Plan(
    goal=plan.goal,
    steps=(plan.steps[2], plan.steps[0], plan.steps[1]),
)
invalid_result = validator.validate(invalid)
valid, invalid_result

In [ ]:
assert valid.valid is True
assert invalid_result.valid is False
assert "patch appears before dependency inspect" in invalid_result.errors

## Ready does not mean done

The plan names intended work. Runtime facts decide which step is ready. Merely becoming ready must not mutate state or mark the step complete.


In [ ]:
scheduler = StepScheduler()
completed = {"reproduce"}
facts = {"failure reproduced"}
ready = scheduler.ready_steps(plan, completed=completed, state_facts=facts)

assert [step.id for step in ready] == ["inspect"]
assert "inspect" not in completed
assert "parser inspected" not in facts
assert plan.version == 1
ready

## Observation invalidates the remaining route

Suppose inspection evidence reveals that delimiter handling actually lives in `csv_adapter.py`. We should preserve the completed reproduction step and replace only the remaining intended work.


In [ ]:
decision = Replanner().replace_remaining(
    plan,
    completed={"reproduce"},
    replacement_steps=(
        PlanStep(
            "inspect_adapter",
            "read_file csv_adapter.py",
            dependencies=("reproduce",),
            preconditions=("failure reproduced",),
            expected_evidence=("adapter inspected",),
        ),
        PlanStep(
            "patch_adapter",
            "edit csv_adapter.py",
            dependencies=("inspect_adapter",),
            preconditions=("adapter inspected",),
            expected_evidence=("source changed",),
        ),
    ),
    reason="evidence showed delimiter handling lives in csv_adapter.py",
)

assert decision.previous.version == 1
assert decision.revised.version == 2
assert decision.preserved_completed == ("reproduce",)
assert decision.removed_remaining == ("inspect", "patch")
assert [step.id for step in decision.revised.steps] == [
    "reproduce",
    "inspect_adapter",
    "patch_adapter",
]
assert [step.id for step in plan.steps] == ["reproduce", "inspect", "patch"]
decision

## Experiment: structural validity is not semantic validity

The finished chapter separates two questions: **is this plan well formed?** and **would this plan actually achieve the goal?** The Stage-04 `PlanValidator` intentionally answers the first question only.

A semantically useless route can therefore be structurally valid. That is not a bug in structural validation; it is the boundary that tells us where semantic plan evaluation belongs.


In [ ]:
semantically_useless = Plan(
    goal="Fix pipe-delimited records",
    steps=(
        PlanStep(
            "announce",
            "announce that the task is complete",
            expected_evidence=("announcement sent",),
        ),
    ),
)

structural_only = validator.validate(semantically_useless)
assert structural_only.valid is True
assert semantically_useless.goal == "Fix pipe-delimited records"
assert semantically_useless.steps[0].action == "announce that the task is complete"

{
    "structurally_valid": structural_only.valid,
    "semantically_achieves_goal": False,
}

## Experiment: infeasibility can be represented without pretending a step ran

A ready-step query can return **no executable next step** because required facts are absent. That is a useful planning/runtime result, not evidence that the missing step executed or failed.


In [ ]:
blocked_plan = Plan(
    goal="Fix pipe-delimited records",
    steps=(
        PlanStep(
            "patch",
            "edit parser.py",
            preconditions=("parser inspected",),
            expected_evidence=("source changed",),
        ),
    ),
)

blocked_ready = scheduler.ready_steps(
    blocked_plan,
    completed=set(),
    state_facts={"failure reproduced"},
)

assert blocked_ready == ()
assert "parser inspected" not in {"failure reproduced"}
blocked_ready

## What was earned

The agent can now represent future work explicitly, validate relationships before execution, choose only steps whose dependencies and current preconditions are satisfied, and revise the remaining route without rewriting history.

**Plan is intended future work; state is what is actually true.** Stage 05 will add progress, budgets, cycles, and named termination around execution.


## Demo API now implemented

```python
from first_principles_agent.planning import (
    Plan, PlanStep, PlanValidator, StepScheduler, Replanner
)
scheduler.ready_steps(plan, completed=..., state_facts=...)
replanner.replace_remaining(plan, completed=..., replacement_steps=..., reason=...)
```
